# Eve SFT — Gemma 4 E2B QLoRA on Kaggle

**Before running:** Settings → Accelerator **GPU T4** (the run uses one; E2B QLoRA fits a single T4 in fp32), Internet **on**.
Both require a phone-verified Kaggle account.

**Input:** attach the `therapylink-corpus` dataset (+ Add Input → Datasets → Your Datasets).
It carries the training and validation splits *and* a copy of the `therapylink_ml` package, so no GitHub clone is needed.

**v16 plan** (docs/ship-review/SHIP_PLAN.md, row 1a/1b): the **v16-clean** run trains only on commercially usable
sources (`--exclude-nc`) for 2 epochs, keeps each epoch's adapter, and gates both. The **v16-full** control cell is
off by default; it exists only to measure what the non-commercial data buys and is never shipped without a legal opinion.

In [ ]:
# transformers>=5.5.2 is required, not optional: on 5.5.0/5.5.1 Gemma 4's E-series shares its
# KV cache through a module-scoped carrier, and gradient checkpointing then corrupts the graph —
# Unsloth refuses to train. Kaggle's image ships 5.5.0. Keeping checkpointing on matters here:
# on a T4 we are in fp32, and it is what keeps activations inside 14.5 GB.
%pip install -q unsloth trl bitsandbytes typer rich pydantic
%pip install -q -U "transformers>=5.5.2"

import glob, os, shutil, subprocess, sys, time

# The v16-full control shares this session; it gets whatever the 12 h limit leaves.
SESSION_START = time.time()

WORK = '/kaggle/working/tl'
os.makedirs(WORK, exist_ok=True)

# Prefer the git repo (easier to iterate); fall back to the copy inside the dataset.
REPO = os.environ.get('TL_REPO', '')
src = None
if REPO:
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK + '/repo'], check=True)
        src = WORK + '/repo/ml'
    except subprocess.CalledProcessError:
        print('clone failed, using the dataset copy')
if src is None:
    hits = glob.glob('/kaggle/input/**/code/therapylink_ml', recursive=True)
    assert hits, 'No therapylink_ml found. Attach the therapylink-corpus dataset.'
    src = os.path.dirname(hits[0])

shutil.copytree(src, WORK + '/ml', dirs_exist_ok=True)
os.chdir(WORK + '/ml')
%pip install -q -e . --no-deps
print('source:', src)

# Kaggle can mount the previous dataset version for a while after an upload reports "ready" (v16's
# first launch trained nothing: the code it found had no --val-size). Fail here, in seconds, if the
# mounted code predates the flags this notebook passes.
_help = subprocess.run([sys.executable, '-m', 'therapylink_ml.train.eve_sft', '--help'], capture_output=True, text=True).stdout
for _flag in ('--val-size', '--min-quality', '--mentalchat-max-words', '--notes-frac'):
    assert _flag in _help, f'stale code in the dataset: eve_sft has no {_flag}. Wait for the new dataset version, then rerun.'
!python -c "import transformers, trl, unsloth; print('transformers', transformers.__version__, '| trl', trl.__version__)"

In [ ]:
train = glob.glob('/kaggle/input/**/conversations.train.jsonl', recursive=True)[0]
os.makedirs('data/processed', exist_ok=True)
shutil.copy(train, 'data/processed/conversations.train.jsonl')
with open(train, encoding='utf-8') as f:
    print(train, sum(1 for _ in f), 'conversations')
# The validation split feeds --val: a stratified 300-conversation subset scored every half epoch.
val = glob.glob('/kaggle/input/**/conversations.val.jsonl', recursive=True)
assert val, 'No conversations.val.jsonl in the corpus dataset; re-push it with tl-kaggle corpus --update'
shutil.copy(val[0], 'data/processed/conversations.val.jsonl')

In [ ]:
# Smoke run: 30 steps on Gemma 4 E2B, ONE GPU. Off by default now that the pipeline is proven
# (v11: loss 2.77 -> 1.91 in 30 steps). Turn it back on after changing the trainer or the image.
#
# CUDA_VISIBLE_DEVICES=0 is load-bearing and stays outside the `if`. Gemma 4's E-series keeps
# per-layer input embeddings and that path indexes the embedding table inside the forward pass, so
# a model sharded across both T4s dies with a cuda:0/cuda:1 mismatch. Unsloth picks device
# placement itself and ignores device_map, so the only reliable way to pin it is to hide the second
# GPU before torch loads.
#
# E2B (~7.5 GB in 4-bit) fits one 14.5 GB T4; E4B (~10.2 GB + activations) does not. E2B is also
# what the app downloads by default, so it is the shipping target regardless.
import subprocess, sys
%env CUDA_VISIBLE_DEVICES=0
%env HF_DATASETS_DISABLE_CACHING=1
RUN_SMOKE = True
if RUN_SMOKE:
    # The v16 trainer's validation passes, per-epoch saves and gate were written without a GPU to
    # run them on. 64 conversations x 2 epochs exercises all three in ~10 minutes, so a bug costs
    # minutes here instead of hours of the real run.
    import itertools
    DATA_DIR = os.path.dirname(glob.glob('/kaggle/input/**/conversations.train.jsonl', recursive=True)[0])
    with open(DATA_DIR + '/conversations.train.jsonl', encoding='utf-8') as f, open('/kaggle/working/smoke.jsonl', 'w', encoding='utf-8') as g:
        g.writelines(itertools.islice(f, 64))
    SMOKE = '/kaggle/working/runs/eve-smoke'
    subprocess.run([sys.executable, '-u', '-m', 'therapylink_ml.train.eve_sft', '--model-name', 'unsloth/gemma-4-E2B-it',
                    '--data', '/kaggle/working/smoke.jsonl', '--val-size', '16', '--epochs', '2', '--batch', '2', '--grad-accum', '2',
                    '--notes-frac', '0.3', '--no-gguf', '--out', SMOKE], check=True)
    for d in ('adapter-epoch1', 'adapter-epoch2', 'adapter'):
        assert os.path.isfile(f'{SMOKE}/{d}/adapter_model.safetensors'), f'smoke run did not save {d}'
    print('smoke OK:', sorted(os.listdir(SMOKE)))
    shutil.rmtree(SMOKE)


In [ ]:
# v16-clean: the candidate that can ship. Fresh from the base model, 2 epochs, commercial-clean data only.
#
# --exclude-nc drops ESConv, EmpatheticDialogues and CounselChat (non-commercial or unverified terms),
#   about half of the v14/v15 mix. What is left: AnnoMI, MentalChat16k and the authored rows.
# --min-quality 1.0 drops AnnoMI transcripts rated low quality (at annomi=3 they were 60 copies an epoch).
# --mentalchat-max-words 180 keeps MentalChat16k only where every reply is short prose; its median
#   reply was 331 words and 21% were lists. mentalchat16k=0.5 applies to what is left.
# tl_authored=5 for 2 epochs keeps each authored row at about 10 exposures (v14: 10 x 1).
# --val-size 300 scores a stratified validation subset every half epoch; each epoch's adapter is kept
#   as adapter-epoch1 / adapter-epoch2, so the checkpoint pick compares them.
#
# Estimated 3-5 h on one T4 (smaller corpus; the first run's train_runtime confirms it).
RUN = '/kaggle/working/runs/eve-sft-e2b-v16-clean'
!python -u -m therapylink_ml.train.eve_sft   --model-name unsloth/gemma-4-E2B-it   --exclude-nc   --weights annomi=3,mentalchat16k=0.5,tl_authored=5   --min-quality 1.0 --mentalchat-max-words 180 --val-size 300   --min-reply-tokens 20 --epochs 2 --lr 2e-4 --warmup 0.05 --notes-frac 0.3   --max-hours 10 --no-gguf   --out {RUN}

In [ ]:
# Gate before download, on every epoch's adapter. gate.json lands next to each adapter.
# A `!command` in a notebook never fails the cell, whatever its exit code: v14's gate printed
# 5/6 and exited 1, and Kaggle still marked the version COMPLETE. The failure is raised in the
# last cell instead, after the v16-full control has had its turn: raising here would stop Run All
# and waste the rest of the session.
import subprocess, sys

GATES = {}

def gate(run):
    adapters = sorted(glob.glob(run + '/adapter-epoch*')) + [run + '/adapter']
    passed = {a: subprocess.run([sys.executable, '-u', '-m', 'therapylink_ml.train.eos_check', a]).returncode == 0
              for a in adapters if os.path.isdir(a)}
    print(run, passed)
    GATES[run] = passed

gate(RUN)


In [ ]:
# v16-full: the control. Same recipe with the non-commercial sources back in (as llm-engineer.md s2
# row A): esconv=0.5, counselchat=1.5, empathetic_dialogues=0. It only measures what that data buys.
# NEVER ship it without a legal opinion on the ESConv / EmpatheticDialogues / CounselChat terms.
# On for this round (SHIP_PLAN row 1b); it gets whatever time the 12 h session has left after v16-clean.
RUN_FULL_CONTROL = True
if RUN_FULL_CONTROL:
    LEFT = f'{max(0.5, 11.3 - (time.time() - SESSION_START) / 3600):.2f}'
    FULL = '/kaggle/working/runs/eve-sft-e2b-v16-full'
    !python -u -m therapylink_ml.train.eve_sft     --model-name unsloth/gemma-4-E2B-it     --weights esconv=0.5,annomi=3,counselchat=1.5,mentalchat16k=0.5,empathetic_dialogues=0,tl_authored=5     --min-quality 1.0 --mentalchat-max-words 180 --val-size 300     --min-reply-tokens 20 --epochs 2 --lr 2e-4 --warmup 0.05 --notes-frac 0.3     --max-hours {LEFT} --no-gguf     --out {FULL}
    gate(FULL)

In [ ]:
# The version fails only when the ship candidate has no passing adapter. The control's result is
# informational. Read every gate.json either way.
print(GATES)
if not any(GATES.get(RUN, {}).values()):
    raise RuntimeError(f'no v16-clean adapter passed the gate; see gate.json in {RUN}')


In [ ]:
# Save Version before closing, or /kaggle/working is wiped.
!ls -la /kaggle/working/runs/*/ 2>/dev/null || ls -la /kaggle/working/runs/